<a href="https://colab.research.google.com/github/EmptyRain75/mnist-ai-classifier/blob/main/notebooks/05_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN for MNIST Classification

## Objective

The previous MLP introduced nonlinear hidden representations, but it still flattened each MNIST image into a 784-dimensional vector.

```text
1 × 28 × 28
    ↓
Flatten
    ↓
784
```

This removes the explicit 2D spatial structure of the image.

A Convolutional Neural Network (CNN) instead processes the image directly:

```text
1 × 28 × 28
      ↓
Convolution
      ↓
Feature Maps
      ↓
Pooling
      ↓
Deeper Features
      ↓
Classifier
      ↓
10 logits
```

In this notebook, we will:

1. Load MNIST while preserving the image shape.
2. Build a basic CNN in PyTorch.
3. Track tensor shapes through the network.
4. Train the CNN using Cross Entropy and SGD.
5. Evaluate train, validation, and test accuracy.
6. Compare the CNN conceptually with the previous MLP.

## Module 1 — Load MNIST

MNIST images are loaded as tensors with shape:

```text
(batch_size, channels, height, width)
```

For MNIST:

```text
N × 1 × 28 × 28
```

where:

- `N` = batch size
- `1` = grayscale channel
- `28 × 28` = image dimensions

Unlike the MLP, the CNN keeps this spatial representation.

In [2]:
import time

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [3]:
def load_mnist(
    batch_size=32,
    val_size=10000,
    seed=42
):
    transform = transforms.ToTensor()

    full_train_dataset = datasets.MNIST(
        root="data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="data",
        train=False,
        download=True,
        transform=transform
    )

    train_size = len(full_train_dataset) - val_size

    train_dataset, val_dataset = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(seed)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader, test_loader

In [5]:
train_loader, val_loader, test_loader = load_mnist()

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Image batch shape: torch.Size([32, 1, 28, 28])
Label batch shape: torch.Size([32])


## Module 2 — CNN Architecture and Shape Tracking

The main limitation of the MLP is that it flattens the image immediately:

```text
1 × 28 × 28
    ↓
Flatten
    ↓
784
```

This removes the explicit 2D spatial structure.

A CNN keeps the image in spatial form and learns local patterns such as edges, curves, and shapes using convolutional filters.

Our baseline CNN will use:

```text
Input
1 × 28 × 28

↓

Conv2d
1 → 32 channels
3 × 3 kernel
padding = 1

↓

32 × 28 × 28

↓

ReLU

↓

MaxPool2d
2 × 2

↓

32 × 14 × 14

↓

Conv2d
32 → 64 channels
3 × 3 kernel
padding = 1

↓

64 × 14 × 14

↓

ReLU

↓

MaxPool2d
2 × 2

↓

64 × 7 × 7

↓

Flatten

↓

3136 features

↓

Linear

↓

10 logits
```

### Convolution

A convolutional layer applies small learnable filters across local regions of the image.

```text
Input image
   ↓
3 × 3 filter slides across image
   ↓
Feature map
```

Important parameters:

- `in_channels` — number of input channels
- `out_channels` — number of learned filters / output feature maps
- `kernel_size` — size of each filter
- `stride` — how far the filter moves each step
- `padding` — pixels added around the image border

For a convolution, the spatial output size is:

```text
output_size = floor((input + 2 × padding - kernel_size) / stride) + 1
```

For our first layer:

```text
input   = 28
kernel  = 3
padding = 1
stride  = 1
```

so:

```text
output = 28
```

Therefore:

```text
1 × 28 × 28
    ↓
Conv2d(1 → 32)
    ↓
32 × 28 × 28
```

### Why Multiple Channels?

Each convolutional filter can learn a different local pattern.

```text
Filter 1 → one type of feature
Filter 2 → another feature
...
Filter 32 → another feature
```

So `out_channels=32` produces 32 feature maps.

### ReLU

ReLU introduces nonlinearity:

```text
ReLU(x) = max(0, x)
```

Without nonlinear activation functions, stacking multiple linear/convolutional layers would still behave like a linear transformation.

### Max Pooling

Max pooling reduces the spatial dimensions.

For:

```text
MaxPool2d(kernel_size=2)
```

each `2 × 2` region becomes one value.

```text
32 × 28 × 28
      ↓
MaxPool2d(2)
      ↓
32 × 14 × 14
```

Pooling:

- reduces computation,
- reduces the size of later feature maps,
- makes the representation less sensitive to small spatial shifts.

### Shape Tracking

The complete shape progression is:

| Layer | Output Shape |
|---|---|
| Input | `1 × 28 × 28` |
| Conv1 | `32 × 28 × 28` |
| ReLU | `32 × 28 × 28` |
| Pool1 | `32 × 14 × 14` |
| Conv2 | `64 × 14 × 14` |
| ReLU | `64 × 14 × 14` |
| Pool2 | `64 × 7 × 7` |
| Flatten | `3136` |
| Linear | `10` |

The key architectural difference is:

```text
MLP
Flatten immediately
    ↓
Learn from individual pixel values

CNN
Convolution first
    ↓
Learn local spatial features
    ↓
Flatten only near the classifier
```

In [6]:
conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=32,
    kernel_size=3,
    padding=1
)

pool = nn.MaxPool2d(kernel_size=2)

x = images

print("Input :", x.shape)

x = conv1(x)
print("Conv1 :", x.shape)

x = pool(x)
print("Pool1 :", x.shape)

Input : torch.Size([32, 1, 28, 28])
Conv1 : torch.Size([32, 32, 28, 28])
Pool1 : torch.Size([32, 32, 14, 14])


## Module 3 — Define the CNN

We now combine the layers from Module 2 into a reusable PyTorch model.

The CNN has two parts:

```text
Feature Extractor
Conv → ReLU → Pool
Conv → ReLU → Pool

        ↓

Classifier
Flatten → Linear → 10 logits
```

The convolutional layers learn spatial features, while the final linear layer maps those features to the 10 MNIST classes.

The model returns **logits**, not probabilities.

`nn.CrossEntropyLoss()` will handle the Softmax operation internally during training.

In [7]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.relu1 = nn.ReLU()

        self.pool1 = nn.MaxPool2d(
            kernel_size=2
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.relu2 = nn.ReLU()

        self.pool2 = nn.MaxPool2d(
            kernel_size=2
        )

        self.fc = nn.Linear(
            64 * 7 * 7,
            num_classes
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = torch.flatten(x, start_dim=1)

        x = self.fc(x)

        return x

In [8]:
model = CNN()

print(model)

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=3136, out_features=10, bias=True)
)


In [9]:
sample_output = model(images)

print("Input shape :", images.shape)
print("Output shape:", sample_output.shape)

Input shape : torch.Size([32, 1, 28, 28])
Output shape: torch.Size([32, 10])


The output shape is:

```text
batch_size × number_of_classes
```

For a batch of 32 MNIST images:

```text
32 × 10
```

Each row contains 10 logits, one for each digit class.

The predicted class is:

```python
prediction = argmax(logits)
```

The model architecture is now complete. The next step is to define the loss function and optimizer.

## Module 4 — Loss and Optimizer

The CNN outputs 10 logits for each image.

For multiclass classification, we use:

```text
Cross Entropy Loss
```

PyTorch's `nn.CrossEntropyLoss()` expects raw logits, so we do **not** apply Softmax manually inside the model.

For optimization, we keep the same simple setup used in the previous models:

```text
Optimizer: SGD
Learning rate: 0.1
```

This keeps the comparison focused on the architectural change from MLP to CNN rather than optimization tricks.

In [10]:
criterion = nn.CrossEntropyLoss()

learning_rate = 0.1

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate
)

In [11]:
logits = model(images)

loss = criterion(
    logits,
    labels
)

print("Logits shape:", logits.shape)
print("Loss:", loss.item())

Logits shape: torch.Size([32, 10])
Loss: 2.2885069847106934


The training objective is now:

```text
Images
  ↓
CNN
  ↓
Logits
  ↓
Cross Entropy
  ↓
Loss
  ↓
Backpropagation
  ↓
SGD update
```

The loss and optimization pipeline is therefore reused from the MLP stage.

Only the model architecture has changed.

## Module 5 — Training Loop

The training loop is almost identical to the MLP stage.

For each batch:

```text
Images
  ↓
Forward pass
  ↓
Logits
  ↓
Cross Entropy Loss
  ↓
Clear old gradients
  ↓
Backpropagation
  ↓
SGD update
```

The important point is that the **training process is reused**.

The main change from the MLP is only the model architecture.

In [12]:
def train_model(
    model,
    train_loader,
    criterion,
    optimizer,
    n_epochs=10
):
    loss_history = []

    start_time = time.perf_counter()

    for epoch in range(n_epochs):
        model.train()

        epoch_loss = 0.0

        for images, labels in train_loader:

            # Forward pass
            logits = model(images)

            # Compute loss
            loss = criterion(
                logits,
                labels
            )

            # Clear old gradients
            optimizer.zero_grad()

            # Backpropagation
            loss.backward()

            # Update parameters
            optimizer.step()

            epoch_loss += loss.item()

        average_loss = (
            epoch_loss / len(train_loader)
        )

        loss_history.append(
            average_loss
        )

        print(
            f"Epoch {epoch + 1}/{n_epochs} | "
            f"Loss: {average_loss:.4f}"
        )

    training_time = (
        time.perf_counter() - start_time
    )

    return loss_history, training_time

In [13]:
n_epochs = 10

loss_history, training_time = train_model(
    model,
    train_loader,
    criterion,
    optimizer,
    n_epochs=n_epochs
)

Epoch 1/10 | Loss: 0.1778
Epoch 2/10 | Loss: 0.0618
Epoch 3/10 | Loss: 0.0467
Epoch 4/10 | Loss: 0.0381
Epoch 5/10 | Loss: 0.0320
Epoch 6/10 | Loss: 0.0272
Epoch 7/10 | Loss: 0.0228
Epoch 8/10 | Loss: 0.0202
Epoch 9/10 | Loss: 0.0181
Epoch 10/10 | Loss: 0.0154


The CNN is now training end-to-end.

During backpropagation, PyTorch automatically computes gradients through:

```text
Linear layer
    ↓
Pooling
    ↓
ReLU
    ↓
Convolution
    ↓
Earlier convolution layers
```

The convolution filters are therefore learned from the classification loss in the same way that the MLP's weights were learned.

No manual gradient derivation is required because PyTorch's autograd handles the backward pass.

## Module 6 — Evaluation

During evaluation, the model should not update its parameters.

We therefore use:

```text
model.eval()
torch.no_grad()
```

For each batch:

```text
Images
  ↓
CNN
  ↓
10 logits
  ↓
argmax
  ↓
Predicted class
```

Accuracy is then calculated by comparing the predictions with the true labels.

In [14]:
def evaluate(model, data_loader):
    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():
        for images, labels in data_loader:

            logits = model(images)

            predictions = torch.argmax(
                logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return correct / total

In [15]:
train_accuracy = evaluate(
    model,
    train_loader
)

val_accuracy = evaluate(
    model,
    val_loader
)

test_accuracy = evaluate(
    model,
    test_loader
)

print("Results")
print("-------")
print(f"Training time      : {training_time:.2f} s")
print(f"Training accuracy  : {train_accuracy:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Test accuracy      : {test_accuracy:.4f}")

Results
-------
Training time      : 355.75 s
Training accuracy  : 0.9968
Validation accuracy: 0.9874
Test accuracy      : 0.9893


The CNN can now be compared with the previous MLP using the same evaluation pipeline.

The main difference is therefore architectural:

```text
MLP
Flatten image immediately
        ↓
Fully connected representation


CNN
Preserve image structure
        ↓
Learn local spatial features
        ↓
Flatten only near the classifier
```

This lets us isolate the effect of replacing a fully connected image representation with a convolutional one.

## Module 7 — MLP vs CNN

The MLP and CNN use the same dataset, loss function, optimizer, and evaluation pipeline.

The main difference is **how the image is represented and processed**.

### MLP

```text
1 × 28 × 28
    ↓
Flatten
    ↓
784
    ↓
Linear
    ↓
ReLU
    ↓
Linear
    ↓
10 logits
```

The MLP treats the image as a flat vector.

It can learn nonlinear relationships, but it does not explicitly use the 2D spatial structure of the image.

### CNN

```text
1 × 28 × 28
    ↓
Convolution
    ↓
Local feature maps
    ↓
Pooling
    ↓
Deeper feature maps
    ↓
Flatten
    ↓
Linear
    ↓
10 logits
```

The CNN preserves the spatial structure for most of the network and learns local patterns using shared convolutional filters.

### Key Difference

```text
MLP
Global fully connected processing

CNN
Local spatial processing + shared filters
```

This gives CNNs a useful inductive bias for image data.

The CNN assumes that:

- nearby pixels are related,
- local patterns are important,
- the same pattern can appear in different parts of the image.

This makes CNNs more suitable for image classification than a basic MLP.

| Model | Input Representation | Main Operation | Spatial Structure |
|---|---|---|---|
| Softmax Regression | Flattened vector | Linear | No |
| MLP | Flattened vector | Fully connected + ReLU | No |
| CNN | 2D image | Convolution + pooling | Yes |

The progression so far is:

```text
Softmax Regression
Linear classifier
        ↓
MLP
Nonlinear representation
        ↓
CNN
Nonlinear + spatial representation
```

The next architectural step will explore a different way of processing images: **Vision Transformer**, which represents an image as a sequence of patches and uses self-attention instead of convolution.

## Conclusion

In this notebook, we extended the MNIST classifier from an MLP to a Convolutional Neural Network.

The main architectural change was:

```text
MLP
Flatten image immediately
        ↓
Fully connected layers

CNN
Preserve 2D image structure
        ↓
Convolutional feature extraction
        ↓
Flatten near the end
        ↓
Classifier
```

The CNN introduced several new components:

```text
Convolution
Feature maps
Channels
ReLU
Pooling
Spatial shape tracking
```

At the same time, much of the previous training pipeline was reused:

```text
MNIST
Train / validation / test split
Cross Entropy
SGD
Backpropagation
Evaluation
```

This demonstrates the modular progression of the project:

```text
Softmax Regression
Linear representation
        ↓
MLP
Nonlinear representation
        ↓
CNN
Nonlinear + spatial representation
```

The CNN is better suited to image data because it explicitly exploits local spatial structure and reuses filters across the image.

The next stage is the **Vision Transformer**, which will replace convolution with a patch-based representation and self-attention.

Further CNN optimization techniques such as Batch Normalization, Adam, data augmentation, residual connections, and other training improvements are intentionally postponed to the `experiments/` stage.